# Analise de Clientes Fieis - Questao 5

## Objetivo
Identificar os 10 clientes mais fieis com base no ticket medio, considerando apenas clientes que possuem diversidade de categorias de produtos (3 ou mais categorias distintas).

## Metricas utilizadas
- **Faturamento Total**: soma da coluna `total` por cliente
- **Frequencia**: contagem de transacoes distintas por cliente
- **Ticket Medio**: Faturamento Total / Frequencia
- **Diversidade**: quantidade de categorias distintas (apos limpeza) por cliente
- **Criterio de desempate**: `id_client` ascendente

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import warnings

warnings.filterwarnings('ignore')

## 1. Carregamento dos Dados

In [2]:
df_vendas = pd.read_csv('../datasets/vendas_2023_2024.csv')
df_produtos = pd.read_csv('../datasets/produtos_raw.csv')

print(f"Vendas: {df_vendas.shape[0]} registros, {df_vendas.shape[1]} colunas")
print(f"Produtos: {df_produtos.shape[0]} registros, {df_produtos.shape[1]} colunas")

print(f"\nColunas vendas: {list(df_vendas.columns)}")
print(f"Colunas produtos: {list(df_produtos.columns)}")

df_vendas.head()

Vendas: 9895 registros, 6 colunas
Produtos: 157 registros, 4 colunas

Colunas vendas: ['id', 'id_client', 'id_product', 'qtd', 'total', 'sale_date']
Colunas produtos: ['name', 'price', 'code', 'actual_category']


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


In [3]:
df_produtos.head(10)

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz
5,Transponder AIS Vector,R$ 11820.21,6,Eletrunicos
6,Radar AIS Zen,R$ 19518.77,7,eLeTrÔnIcOs
7,GPS AIS Zen,R$ 4984.15,8,E L E T R Ô N I C O S
8,Transponder AIS Titan Pulse,R$ 39705.5,9,Eletronicoz
9,Piloto Automático Simrad Titan Flux Magnum,R$ 32033.04,10,eletrônicos


## 2. Limpeza de Categorias

As categorias no dataset de produtos apresentam inconsistencias (espacos, acentuacao, variacao de caixa). A padronizacao segue a regra:
- Remover espacos e converter para minusculas
- Remover acentos para facilitar a comparacao
- Classificar com base em padroes de texto

In [4]:
print(f"Categorias originais (distintas): {df_produtos['actual_category'].nunique()}")
print(f"\nExemplos de categorias antes da limpeza:")
print(df_produtos['actual_category'].value_counts())

Categorias originais (distintas): 39

Exemplos de categorias antes da limpeza:
actual_category
AncorageM                9
Propução                 8
Ancoraguem               8
eletrônicos              7
Eletronicoz              7
E L E T R Ô N I C O S    6
ELETRONICOS              6
P R O P U L S Ã O        6
propulsão                6
PROPULSAO                6
Eletrunicos              5
Ancorajm                 5
eLeTrÔnIcOs              5
Propulssão               5
Propulção                5
Prop                     5
Encoragem                5
aNcOrAgEm                5
A N C O R A G E M        5
Eletrônicos              4
propulsao                4
EletrônicoS              3
Propulçao                3
Ancorajem                3
eletronicos              3
Ancoragem                3
Ancorajen                2
Eletronicos              2
Propulsam                2
Eletroniscos             2
pRoPuLsÃo                2
ancoragem                2
AnCoRaGeM                2
ELEtRÔNICOS   

In [5]:
def clean_category(cat):
    cat = cat.replace(' ', '').lower()
    cat = unicodedata.normalize('NFKD', cat).encode('ascii', 'ignore').decode()
    if 'eletr' in cat:
        return 'eletrônicos'
    elif 'prop' in cat:
        return 'propulsão'
    elif 'ancor' in cat or 'encor' in cat:
        return 'ancoragem'
    return cat


df_produtos['categoria'] = df_produtos['actual_category'].apply(clean_category)

print("Categorias apos limpeza:")
print(df_produtos['categoria'].value_counts())

Categorias apos limpeza:
categoria
propulsão      53
ancoragem      53
eletrônicos    51
Name: count, dtype: int64


## 3. Remocao de Duplicatas de Produtos

O dataset de produtos possui registros duplicados por `code`. Manteremos apenas a primeira ocorrencia.

In [6]:
print(f"Produtos antes da deduplicacao: {len(df_produtos)}")
df_produtos = df_produtos.drop_duplicates(subset='code', keep='first').reset_index(drop=True)
print(f"Produtos apos deduplicacao: {len(df_produtos)}")

Produtos antes da deduplicacao: 157
Produtos apos deduplicacao: 150


## 4. Juncao de Vendas com Produtos

In [7]:
df = df_vendas.merge(
    df_produtos[['code', 'categoria']],
    left_on='id_product',
    right_on='code',
    how='left'
)

print(f"Registros apos juncao: {len(df)}")
print(f"Registros sem categoria: {df['categoria'].isna().sum()}")
df[['id', 'id_client', 'id_product', 'qtd', 'total', 'categoria']].head(10)

Registros apos juncao: 9895
Registros sem categoria: 0


,id,id_client,id_product,qtd,total,categoria
0,0,42,105,11,3405.00,ancoragem
1,1,3,136,9,16873.90,ancoragem
2,2,25,139,7,9475.30,ancoragem
3,4,20,23,5,55893.00,eletrônicos
4,5,8,57,4,451403.90,propulsão
5,6,36,52,3,39056.40,propulsão
6,8,27,25,3,34560.05,eletrônicos
7,9,37,26,7,114932.90,eletrônicos
8,10,31,143,3,12643.55,ancoragem
9,11,39,128,5,23254.00,ancoragem


## 5. Calculo de Metricas por Cliente

In [8]:
df_clientes = (
    df
    .groupby('id_client')
    .agg(
        faturamento_total=('total', 'sum'),
        frequencia=('id', 'nunique'),
        diversidade_categorias=('categoria', 'nunique')
    )
    .reset_index()
)

df_clientes['ticket_medio'] = (
    df_clientes['faturamento_total'] / df_clientes['frequencia']
).round(2)

print(f"Total de clientes: {len(df_clientes)}")
print(f"\nDistribuicao de diversidade de categorias:")
print(df_clientes['diversidade_categorias'].value_counts().sort_index())

df_clientes.head(10)

Total de clientes: 49

Distribuicao de diversidade de categorias:
diversidade_categorias
3    49
Name: count, dtype: int64


,id_client,faturamento_total,frequencia,diversidade_categorias,ticket_medio
0,1,51092500.05,190,3,268907.89
1,2,65652931.35,220,3,298422.42
2,3,59575349.10,207,3,287803.62
3,4,50691754.40,207,3,244887.70
4,5,58592802.70,202,3,290063.38
5,6,53225818.95,201,3,264805.07
6,7,47507921.80,207,3,229506.87
7,8,52156129.50,208,3,250750.62
8,9,66788855.35,218,3,306370.90
9,10,51674436.80,203,3,254553.88


## 6. Filtragem por Diversidade e Ranking

In [9]:
df_filtrado = df_clientes[df_clientes['diversidade_categorias'] >= 3].copy()

print(f"Clientes com 3+ categorias distintas: {len(df_filtrado)}")

df_filtrado = df_filtrado.sort_values(
    ['ticket_medio', 'id_client'],
    ascending=[False, True]
).reset_index(drop=True)

print(f"\nRanking completo dos clientes com diversidade >= 3:")
df_filtrado

Clientes com 3+ categorias distintas: 49

Ranking completo dos clientes com diversidade >= 3:


,id_client,faturamento_total,frequencia,diversidade_categorias,ticket_medio
0,47,64003343.75,190,3,336859.70
1,42,72187369.50,222,3,325168.33
2,9,66788855.35,218,3,306370.90
3,22,59581398.75,198,3,300916.16
4,2,65652931.35,220,3,298422.42
5,28,60826837.25,204,3,298170.77
6,46,59126834.35,199,3,297119.77
7,38,57093331.15,195,3,292786.31
8,36,62791038.15,215,3,292051.34
9,5,58592802.70,202,3,290063.38


In [10]:
top_10 = df_filtrado.head(10)

print("=" * 70)
print("TOP 10 CLIENTES FIEIS (por ticket medio, diversidade >= 3)")
print("=" * 70)

for idx, row in top_10.iterrows():
    print(
        f"Cliente {int(row['id_client']):>3d} | "
        f"Ticket Medio: R$ {row['ticket_medio']:>12,.2f} | "
        f"Faturamento: R$ {row['faturamento_total']:>14,.2f} | "
        f"Frequencia: {int(row['frequencia']):>3d} | "
        f"Categorias: {int(row['diversidade_categorias'])}"
    )

ids_top10 = top_10['id_client'].tolist()
print(f"\nIDs dos Top 10: {ids_top10}")

TOP 10 CLIENTES FIEIS (por ticket medio, diversidade >= 3)
Cliente  47 | Ticket Medio: R$   336,859.70 | Faturamento: R$  64,003,343.75 | Frequencia: 190 | Categorias: 3
Cliente  42 | Ticket Medio: R$   325,168.33 | Faturamento: R$  72,187,369.50 | Frequencia: 222 | Categorias: 3
Cliente   9 | Ticket Medio: R$   306,370.90 | Faturamento: R$  66,788,855.35 | Frequencia: 218 | Categorias: 3
Cliente  22 | Ticket Medio: R$   300,916.16 | Faturamento: R$  59,581,398.75 | Frequencia: 198 | Categorias: 3
Cliente   2 | Ticket Medio: R$   298,422.42 | Faturamento: R$  65,652,931.35 | Frequencia: 220 | Categorias: 3
Cliente  28 | Ticket Medio: R$   298,170.77 | Faturamento: R$  60,826,837.25 | Frequencia: 204 | Categorias: 3
Cliente  46 | Ticket Medio: R$   297,119.77 | Faturamento: R$  59,126,834.35 | Frequencia: 199 | Categorias: 3
Cliente  38 | Ticket Medio: R$   292,786.31 | Faturamento: R$  57,093,331.15 | Frequencia: 195 | Categorias: 3
Cliente  36 | Ticket Medio: R$   292,051.34 | Faturam

## 7. Categoria com Mais Itens Vendidos (Top 10 Clientes)

In [11]:
df_top10_vendas = df[df['id_client'].isin(ids_top10)].copy()

categoria_itens = (
    df_top10_vendas
    .groupby('categoria')['qtd']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'qtd': 'total_itens'})
)

print("Itens vendidos por categoria (Top 10 clientes):")
print(categoria_itens)

categoria_mais_itens = categoria_itens.iloc[0]
print(f"\nCategoria com mais itens vendidos: {categoria_mais_itens['categoria']}")
print(f"Total de itens: {int(categoria_mais_itens['total_itens'])}")

Itens vendidos por categoria (Top 10 clientes):
     categoria  total_itens
0    propulsão         6030
1    ancoragem         5632
2  eletrônicos         5214

Categoria com mais itens vendidos: propulsão
Total de itens: 6030


## 8. Respostas as Questoes de Validacao

In [12]:
print("=" * 70)
print("RESPOSTAS - QUESTAO 5")
print("=" * 70)

print(f"\nQuestao 5.2 - Qual categoria possui mais itens vendidos entre os Top 10?")
print(f"Resposta: {categoria_mais_itens['categoria']}")
print(f"Total de itens: {int(categoria_mais_itens['total_itens'])}")

print(f"\nTop 10 clientes (id_client): {ids_top10}")

RESPOSTAS - QUESTAO 5

Questao 5.2 - Qual categoria possui mais itens vendidos entre os Top 10?
Resposta: propulsão
Total de itens: 6030

Top 10 clientes (id_client): [47, 42, 9, 22, 2, 28, 46, 38, 36, 5]


## SQL de Referencia (Questao 5.1)

```sql
-- SQL Reference (Questao 5.1)
WITH produtos_limpos AS (
    SELECT
        code AS product_id,
        name,
        CASE
            WHEN LOWER(REPLACE(actual_category, ' ', '')) LIKE '%eletr%'
                THEN 'eletrônicos'
            WHEN LOWER(REPLACE(actual_category, ' ', '')) LIKE '%propuls%'
                OR LOWER(REPLACE(actual_category, ' ', '')) LIKE '%prop%'
                THEN 'propulsão'
            WHEN LOWER(REPLACE(actual_category, ' ', '')) LIKE '%ancor%'
                OR LOWER(REPLACE(actual_category, ' ', '')) LIKE '%encor%'
                THEN 'ancoragem'
            ELSE LOWER(TRIM(actual_category))
        END AS categoria
    FROM produtos_raw
    GROUP BY code, name, actual_category
),

metricas_cliente AS (
    SELECT
        v.id_client,
        SUM(v.total) AS faturamento_total,
        COUNT(DISTINCT v.id) AS frequencia,
        ROUND(SUM(v.total) / COUNT(DISTINCT v.id), 2) AS ticket_medio,
        COUNT(DISTINCT p.categoria) AS diversidade_categorias
    FROM vendas v
    LEFT JOIN produtos_limpos p ON v.id_product = p.product_id
    GROUP BY v.id_client
    HAVING COUNT(DISTINCT p.categoria) >= 3
),

top_10 AS (
    SELECT id_client
    FROM metricas_cliente
    ORDER BY ticket_medio DESC, id_client ASC
    LIMIT 10
)

SELECT
    p.categoria,
    SUM(v.qtd) AS total_itens
FROM vendas v
INNER JOIN top_10 t ON v.id_client = t.id_client
LEFT JOIN produtos_limpos p ON v.id_product = p.product_id
GROUP BY p.categoria
ORDER BY total_itens DESC;
```

## Interpretacao dos Resultados (Questao 5.3)

### Metodologia aplicada

1. **Limpeza de categorias**: Removemos espacos, convertemos para minusculas, removemos acentos, e classificamos com base em padroes de texto (contem 'eletr' -> eletronicos, 'prop' -> propulsao, 'ancor'/'encor' -> ancoragem).

2. **Filtragem por diversidade**: Agrupamos por `id_client`, contamos categorias distintas com `COUNT(DISTINCT categoria)`, e filtramos apenas clientes com 3 ou mais categorias.

3. **Contagem apenas dos Top 10**: Primeiro identificamos os 10 clientes com maior ticket medio (entre os que possuem 3+ categorias), depois filtramos as vendas apenas desses clientes para contar os itens por categoria.